# Agent Streaming Events — `@azure/ai-projects`

This sample demonstrates how to create an agent, create a conversation, and stream responses using the agent with event handling.

It mirrors the [`agentStreamEvents.ts`](./agentStreamEvents.ts) sample and runs the **locally built** `@azure/ai-projects` from this repo.

## Prerequisites

1. Build the package: run `pnpm build` in `sdk/ai/ai-projects/`.
2. Install and register the **tslab** kernel (`npm install -g tslab && tslab install`), then select the **TypeScript** kernel for this notebook.
3. Launch VS Code / Jupyter from `sdk/ai/ai-projects/`.
4. Authenticate with `az login`.
5. **Environment variables** — the sample reads these via `process.env`:
   - `FOUNDRY_PROJECT_ENDPOINT` — your Foundry project endpoint.
   - `FOUNDRY_MODEL_NAME` — your model deployment name.

Run the cells in order (top to bottom); state is shared across cells.

In [1]:
import { DefaultAzureCredential } from "@azure/identity";
import { AIProjectClient } from "@azure/ai-projects";

const projectEndpoint = process.env["FOUNDRY_PROJECT_ENDPOINT"] ?? "<project endpoint>";
const deploymentName = process.env["FOUNDRY_MODEL_NAME"] ?? "<model deployment name>";

console.log(`Deployment name: ${deploymentName}`);

Deployment name: gpt-5.2


In [2]:
// Create AI Project client
const project = new AIProjectClient(projectEndpoint, new DefaultAzureCredential());
// Annotated as `any` so tslab does not try to emit a non-portable declaration
// referencing the deep `node_modules/openai` (pnpm junction) path.
const openAIClient: any = project.getOpenAIClient();

In [3]:
// Create agent
console.log("Creating agent...");
const agent = await project.agents.createVersion("MyAgent", {
  kind: "prompt",
  model: deploymentName,
  instructions: "You are a helpful assistant that answers general questions",
});
console.log(`Agent created (id: ${agent.id}, name: ${agent.name}, version: ${agent.version})`);

Creating agent...
Agent created (id: MyAgent:265, name: MyAgent, version: 265)


In [4]:
// Create conversation with initial user message
console.log("\nCreating conversation with initial user message...");
const conversation = await openAIClient.conversations.create({
  items: [{ type: "message", role: "user", content: "Tell me about the capital city of France" }],
});
console.log(`Created conversation with initial user message (id: ${conversation.id})`);


Creating conversation with initial user message...
Created conversation with initial user message (id: conv_6dc1acb7626820cd00kA5Xkfd9fLQlxk8TBBhTSEPlFTdoUnLh)


In [5]:
// Generate streaming response using the agent
console.log("\nGenerating streaming response...");
const stream = openAIClient.responses.stream(
  {
    conversation: conversation.id,
  },
  {
    body: { agent_reference: { name: agent.name, type: "agent_reference" } },
  },
);

// Process streaming events as they arrive
const iterator = stream[Symbol.asyncIterator]();
let next = await iterator.next();
while (!next.done) {
  const event = next.value;
  if (event.type === "response.created") {
    console.log(`Stream response created with ID: ${event.response.id}\n`);
  } else if (event.type === "response.output_text.delta") {
    process.stdout.write(event.delta);
  } else if (event.type === "response.output_text.done") {
    console.log(`\n\nResponse text done. Access final text in 'event.text'`);
  } else if (event.type === "response.completed") {
    console.log(`\nResponse completed. Access final text in 'event.response.output_text'`);
  }
  next = await iterator.next();
}


Generating streaming response...
Stream response created with ID: resp_6dc1acb7626820cd006a5fdd8d5a348190a745b0082fe9eab8

The capital city of France is **Paris**.

- **Location:** Northern France, on the River Seine.  
- **Role:** It’s France’s main center for **government, economy, culture, and education**.  
- **Notable landmarks:** **Eiffel Tower**, **Louvre Museum**, **Notre-Dame Cathedral**, **Arc de Triomphe**, and **Champs-Élysées**.  
- **Reputation:** Often called the **“City of Light”** and known worldwide for art, fashion, cuisine, and architecture.

Response text done. Access final text in 'event.text'

Response completed. Access final text in 'event.response.output_text'


In [6]:
// Clean up
console.log("\nCleaning up resources...");
await openAIClient.conversations.delete(conversation.id);
console.log("Conversation deleted");

await project.agents.deleteVersion(agent.name, agent.version);
console.log("Agent deleted");


Cleaning up resources...
Conversation deleted
Agent deleted
